# 06 - Detect + track on video

Runs YOLO11n detection with **Ultralytics ByteTrack** persistent IDs over a video, overlays boxes + track IDs + **live end-to-end FPS**, and writes an annotated output mp4.

Part 1 reassembles a LettuceMOTS sequence into an mp4 (via ffmpeg) so you have a test clip. Part 2 runs the tracker on any video.

> Ships **unrun** - needs a trained checkpoint (03/04) and a source video. FPS here is measured around the **full** per-frame loop (read + inference + tracking + draw + write), not the model call alone.

In [ ]:
import os, sys
from pathlib import Path
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "croprow" / "utils.py").is_file())
sys.path.insert(0, str(REPO_ROOT))
from croprow import utils as U
CW = REPO_ROOT / "croprow"
DATA_DIR = CW / "data"
MODELS_DIR = CW / "models"
RUNS_DIR = CW / "runs"
RESULTS_MD = CW / "RESULTS.md"
import cv2

# ===================== CONFIG (edit here only) =====================
WEIGHTS       = str(MODELS_DIR / "best.pt")
IMGSZ         = 640
CONF          = 0.25
IOU           = 0.50
TRACKER       = "bytetrack.yaml"    # Ultralytics ByteTrack config
DEVICE        = 0

# --- Part 1: reassemble a LettuceMOTS sequence into an mp4 for testing ---
LETTUCE_ROOT  = os.environ.get("LETTUCE_ROOT", r"D:\croprow_dataset\LettuceMOTS")
SEQ_SPLIT     = "test"              # "train" or "test"
SEQ_ID        = "0003"             # sequence folder to turn into a clip
REASSEMBLE_FPS = 10
REASSEMBLED_MP4 = str(RUNS_DIR / f"seq_{SEQ_SPLIT}_{SEQ_ID}.mp4")

# --- Part 2: tracking source + output ---
SOURCE_VIDEO  = REASSEMBLED_MP4     # or any mp4 path (e.g. your own footage)
OUTPUT_VIDEO  = str(RUNS_DIR / "tracked_output.mp4")
OUT_FPS       = REASSEMBLE_FPS
# ===================================================================
RUNS_DIR.mkdir(parents=True, exist_ok=True)
print("weights:", WEIGHTS)
print("source :", SOURCE_VIDEO)

## Part 1 - reassemble a sequence into mp4 (ffmpeg)

Frames are named `000000.png, 000001.png, ...`; ffmpeg stitches them at `REASSEMBLE_FPS`. Requires `ffmpeg` on PATH.

In [ ]:
import shutil, subprocess
frames_dir = Path(LETTUCE_ROOT) / SEQ_SPLIT / "images" / SEQ_ID
if not frames_dir.is_dir():
    raise FileNotFoundError(f"sequence frames not found: {frames_dir}")
if shutil.which("ffmpeg") is None:
    raise RuntimeError("ffmpeg not found on PATH. Install ffmpeg to reassemble "
                       "sequences, or set SOURCE_VIDEO to an existing mp4.")

first = sorted(frames_dir.glob("*.png"))[0]
cmd = [
    "ffmpeg", "-y",
    "-framerate", str(REASSEMBLE_FPS),
    "-start_number", first.stem,
    "-i", str(frames_dir / "%06d.png"),
    "-c:v", "libx264", "-pix_fmt", "yuv420p",
    REASSEMBLED_MP4,
]
print(" ".join(cmd))
subprocess.run(cmd, check=True)
print("wrote", REASSEMBLED_MP4)

## Part 2 - detect + ByteTrack + annotate

Manual OpenCV loop so the FPS timer wraps the entire pipeline. `persist=True` keeps track IDs stable across frames.

In [ ]:
# This notebook needs the training/inference stack (torch + ultralytics),
# NOT installed in the light 01/02 env. Install into a Python 3.11 venv with
# numpy<2 -- see croprow/requirements-train.txt and croprow/README.md.
try:
    import torch
    from ultralytics import YOLO
    import ultralytics
    print("torch      :", torch.__version__)
    print("ultralytics:", ultralytics.__version__)
    print("CUDA avail :", torch.cuda.is_available(),
          "|", (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"))
except ModuleNotFoundError as e:
    raise ModuleNotFoundError(
        f"Missing training dependency: {e.name}. Install croprow/requirements-train.txt "
        "into a Python 3.11 (numpy<2) venv before running this notebook."
    ) from e

In [ ]:
import time
import numpy as np

if not Path(WEIGHTS).is_file():
    raise FileNotFoundError(f"Weights not found: {WEIGHTS}. Train first (03/04).")
if not Path(SOURCE_VIDEO).is_file():
    raise FileNotFoundError(f"Source video not found: {SOURCE_VIDEO}. Run Part 1 "
                            "or point SOURCE_VIDEO at an mp4.")

model = YOLO(WEIGHTS)
cap = cv2.VideoCapture(SOURCE_VIDEO)
W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
writer = cv2.VideoWriter(OUTPUT_VIDEO, cv2.VideoWriter_fourcc(*"mp4v"), OUT_FPS, (W, H))

n, fps_ema, t_all = 0, None, 0.0
while True:
    ok, frame = cap.read()
    if not ok:
        break
    t0 = time.perf_counter()

    res = model.track(frame, persist=True, tracker=TRACKER, imgsz=IMGSZ,
                      conf=CONF, iou=IOU, device=DEVICE, verbose=False)[0]

    boxes = res.boxes
    if boxes is not None and boxes.xyxy is not None:
        xyxy = boxes.xyxy.cpu().numpy().astype(int)
        ids = (boxes.id.cpu().numpy().astype(int)
               if boxes.id is not None else [-1] * len(xyxy))
        for (x1, y1, x2, y2), tid in zip(xyxy, ids):
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
            label = f"ID {tid}" if tid >= 0 else "lettuce"
            cv2.putText(frame, label, (x1, max(0, y1 - 5)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1, cv2.LINE_AA)

    dt = time.perf_counter() - t0            # FULL loop: infer + track + draw
    t_all += dt
    inst = 1.0 / dt if dt > 0 else 0.0
    fps_ema = inst if fps_ema is None else 0.9 * fps_ema + 0.1 * inst
    cv2.putText(frame, f"FPS {fps_ema:5.1f}", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 255), 2, cv2.LINE_AA)

    writer.write(frame)
    n += 1

cap.release()
writer.release()
mean_fps = n / t_all if t_all > 0 else 0.0
print(f"frames={n}  mean end-to-end FPS={mean_fps:.1f}")
print("wrote", OUTPUT_VIDEO)

Mean end-to-end FPS above is the honest number for this machine + config. See `07_speed` to sweep imgsz and exported backends against the 30 FPS target.